In [1]:
import duckdb
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

sns.set_theme(style="ticks")
mm = 1 / 25.4  # centimeters in inches

import sys

sys.path.append("..")

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()
import numpy as np

In [2]:
PD_patients = ["PD0086", "PD0945", "PD0612", "PD0913", "PD0917"]
HC_patients = ["C030", "C064", "C036", "C043", "C049", "C074"]

In [3]:
import duckdb
import pandas as pd
import numpy as np
import os

# ============================================================================
# USER INPUT: Define specific patients you want to analyze
# ============================================================================

# Define your PD and HC patient lists (replace with your actual patient IDs)
PD_patients = ["PD0086", "PD0945", "PD0612", "PD0913", "PD0917"]
HC_patients = ["C030", "C064", "C036", "C043", "C049", "C074"]

# ============================================================================
# SETUP
# ============================================================================

output_dir = (
    "/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots"
)
os.makedirs(output_dir, exist_ok=True)

# Connect to the parquet database which contains image_filename
parquet_path = "/scratch/duckdb-database/spot_analysis.parquet"
conn = duckdb.connect(database=":memory:")

# ============================================================================
# FUNCTION TO GET IMAGE FILENAMES
# ============================================================================


def get_image_filenames_for_patients(
    conn, patient_list, disease_label, cell_types=None, regions=None
):
    """Get unique image filenames for specified patients"""

    if not patient_list:
        print(f"No patients specified for {disease_label}")
        return pd.DataFrame()

    patient_ids_str = ",".join([f"'{pid}'" for pid in patient_list])

    # Build the WHERE clause
    where_clauses = [f"donorid IN ({patient_ids_str})"]

    if cell_types:
        cell_type_str = ",".join([f"'{ct}'" for ct in cell_types])
        where_clauses.append(f"cell_type IN ({cell_type_str})")

    if regions:
        region_str = ",".join([f"'{r}'" for r in regions])
        where_clauses.append(f"region IN ({region_str})")

    where_clause = " AND ".join(where_clauses)

    # Query to get unique image filenames
    query = f"""
    SELECT DISTINCT
        donorid,
        disease,
        cell_type,
        region,
        image_filename,
        COUNT(*) as oligomer_count_in_image
    FROM '{parquet_path}'
    WHERE {where_clause}
    GROUP BY donorid, disease, cell_type, region, image_filename
    ORDER BY donorid, cell_type, region, image_filename
    """

    try:
        image_data = conn.execute(query).fetchdf()
        return image_data
    except Exception as e:
        print(f"Error querying image filenames: {e}")
        return pd.DataFrame()


# ============================================================================
# MAIN SCRIPT: GET IMAGE FILENAMES
# ============================================================================

print("=" * 80)
print("EXTRACTING RAW IMAGE FILENAMES FOR SPECIFIED PATIENTS")
print("=" * 80)

# Define cell types and regions of interest (optional - can be None for all)
cell_types_of_interest = ["microglia", "neurons"]
regions_of_interest = ["cingulate", "parietal", "frontal"]

all_image_data = pd.DataFrame()

# Process PD patients
if PD_patients:
    print(
        f"\n1. Getting image filenames for PD patients ({len(PD_patients)} patients)..."
    )
    pd_image_data = get_image_filenames_for_patients(
        conn,
        PD_patients,
        "PD",
        cell_types=cell_types_of_interest,
        regions=regions_of_interest,
    )

    if not pd_image_data.empty:
        print(f"   Found {len(pd_image_data)} unique image files for PD patients")
        all_image_data = pd.concat([all_image_data, pd_image_data], ignore_index=True)
    else:
        print("   No image data found for PD patients")

# Process HC patients
if HC_patients:
    print(
        f"\n2. Getting image filenames for HC patients ({len(HC_patients)} patients)..."
    )
    hc_image_data = get_image_filenames_for_patients(
        conn,
        HC_patients,
        "HC",
        cell_types=cell_types_of_interest,
        regions=regions_of_interest,
    )

    if not hc_image_data.empty:
        print(f"   Found {len(hc_image_data)} unique image files for HC patients")
        all_image_data = pd.concat([all_image_data, hc_image_data], ignore_index=True)
    else:
        print("   No image data found for HC patients")

# ============================================================================
# SAVE RESULTS
# ============================================================================

if not all_image_data.empty:
    print(f"\n3. Total unique image files found: {len(all_image_data)}")

    # Save all image data
    all_images_file = f"{output_dir}/raw_image_filenames_all.csv"
    all_image_data.to_csv(all_images_file, index=False)
    print(f"   Saved all image data to: {all_images_file}")

    # Create summary statistics
    print(f"\n4. Summary statistics:")

    # By disease
    print("   By disease:")
    disease_summary = (
        all_image_data.groupby("disease")
        .agg(
            {
                "image_filename": "nunique",
                "donorid": pd.Series.nunique,
                "oligomer_count_in_image": "sum",
            }
        )
        .reset_index()
    )
    disease_summary.columns = [
        "disease",
        "unique_images",
        "unique_patients",
        "total_oligomers",
    ]
    print(disease_summary.to_string(index=False))

    # By cell type
    print("\n   By cell type:")
    celltype_summary = (
        all_image_data.groupby(["disease", "cell_type"])
        .agg({"image_filename": "nunique", "oligomer_count_in_image": "sum"})
        .reset_index()
    )
    print(celltype_summary.to_string(index=False))

    # By region
    print("\n   By region:")
    region_summary = (
        all_image_data.groupby(["disease", "region"])
        .agg({"image_filename": "nunique", "oligomer_count_in_image": "sum"})
        .reset_index()
    )
    print(region_summary.to_string(index=False))

    # Save summaries
    disease_summary.to_csv(f"{output_dir}/image_summary_by_disease.csv", index=False)
    celltype_summary.to_csv(f"{output_dir}/image_summary_by_celltype.csv", index=False)
    region_summary.to_csv(f"{output_dir}/image_summary_by_region.csv", index=False)

    # Create a simplified list of just image filenames
    unique_filenames = all_image_data["image_filename"].unique()
    filename_list_file = f"{output_dir}/unique_image_filenames.txt"
    with open(filename_list_file, "w") as f:
        f.write(f"# Total unique image filenames: {len(unique_filenames)}\n")
        f.write(f"# Generated: {pd.Timestamp.now()}\n")
        f.write("#" * 80 + "\n\n")
        for filename in sorted(unique_filenames):
            f.write(f"{filename}\n")
    print(f"\n   Saved list of unique filenames to: {filename_list_file}")

    # Create per-patient image lists
    print(f"\n5. Creating per-patient image lists...")
    for donorid in all_image_data["donorid"].unique():
        patient_images = all_image_data[all_image_data["donorid"] == donorid]
        disease = (
            patient_images["disease"].iloc[0]
            if "disease" in patient_images.columns
            else "Unknown"
        )

        # Save per-patient CSV
        patient_file = f"{output_dir}/images_{donorid}_{disease}.csv"
        patient_images.to_csv(patient_file, index=False)

        # Create per-patient text file
        patient_txt_file = f"{output_dir}/images_{donorid}_{disease}.txt"
        with open(patient_txt_file, "w") as f:
            f.write(f"# Patient: {donorid} ({disease})\n")
            f.write(
                f"# Total unique images: {patient_images['image_filename'].nunique()}\n"
            )
            f.write(
                f"# Total oligomers in images: {patient_images['oligomer_count_in_image'].sum()}\n"
            )
            f.write("#" * 60 + "\n\n")
            for _, row in patient_images.iterrows():
                f.write(f"Image: {row['image_filename']}\n")
                f.write(f"  Cell type: {row['cell_type']}\n")
                f.write(f"  Region: {row['region']}\n")
                f.write(f"  Oligomers: {row['oligomer_count_in_image']}\n")
                f.write("-" * 40 + "\n")

    print(f"   Created per-patient files in {output_dir}")

    # Create Excel file with multiple sheets
    excel_file = f"{output_dir}/image_filename_analysis.xlsx"
    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        # All image data
        all_image_data.to_excel(writer, sheet_name="All_Image_Data", index=False)

        # Summaries
        disease_summary.to_excel(writer, sheet_name="Summary_by_Disease", index=False)
        celltype_summary.to_excel(writer, sheet_name="Summary_by_CellType", index=False)
        region_summary.to_excel(writer, sheet_name="Summary_by_Region", index=False)

        # Pivot tables
        pivot_celltype = all_image_data.pivot_table(
            index=["donorid", "disease"],
            columns="cell_type",
            values="image_filename",
            aggfunc=lambda x: ", ".join(sorted(set(x))),
            fill_value="",
        ).reset_index()
        pivot_celltype.to_excel(
            writer, sheet_name="Image_List_by_CellType", index=False
        )

        pivot_region = all_image_data.pivot_table(
            index=["donorid", "disease"],
            columns="region",
            values="image_filename",
            aggfunc=lambda x: ", ".join(sorted(set(x))),
            fill_value="",
        ).reset_index()
        pivot_region.to_excel(writer, sheet_name="Image_List_by_Region", index=False)

    print(f"\n   Saved comprehensive Excel file to: {excel_file}")

    # ============================================================================
    # ADDITIONAL ANALYSIS: Extract patterns from filenames
    # ============================================================================

    print(f"\n6. Analyzing filename patterns...")

    # Extract potential patterns (e.g., slide numbers, coordinates, etc.)
    filename_patterns = {}
    for filename in unique_filenames:
        # Common patterns to extract
        parts = filename.split("_")
        if len(parts) > 1:
            # Try to identify slide/scan identifiers
            for part in parts:
                if any(char.isdigit() for char in part):
                    if part not in filename_patterns:
                        filename_patterns[part] = 0
                    filename_patterns[part] += 1

    # Sort patterns by frequency
    sorted_patterns = sorted(
        filename_patterns.items(), key=lambda x: x[1], reverse=True
    )

    # Save pattern analysis
    pattern_file = f"{output_dir}/filename_pattern_analysis.txt"
    with open(pattern_file, "w") as f:
        f.write("FILENAME PATTERN ANALYSIS\n")
        f.write("=" * 60 + "\n")
        f.write(f"Total unique filenames analyzed: {len(unique_filenames)}\n\n")
        f.write("Most common patterns in filenames:\n")
        f.write("-" * 40 + "\n")
        for pattern, count in sorted_patterns[:20]:  # Top 20 patterns
            f.write(
                f"{pattern}: {count} files ({count/len(unique_filenames)*100:.1f}%)\n"
            )

        # Show sample filenames
        f.write(f"\nSample filenames (first 10):\n")
        f.write("-" * 40 + "\n")
        for filename in sorted(unique_filenames)[:10]:
            f.write(f"{filename}\n")

    print(f"   Saved pattern analysis to: {pattern_file}")

    # ============================================================================
    # CREATE BASH SCRIPT FOR FILE COPYING (if needed)
    # ============================================================================

    # Create a bash script to copy these files (if they exist somewhere)
    bash_script = f"{output_dir}/copy_image_files.sh"
    with open(bash_script, "w") as f:
        f.write("#!/bin/bash\n")
        f.write("# Script to copy image files for analysis\n")
        f.write(f"# Generated: {pd.Timestamp.now()}\n")
        f.write("# Total files to copy: {}\n".format(len(unique_filenames)))
        f.write("\n")
        f.write("# Create output directory\n")
        f.write('OUTPUT_DIR="${1:-./copied_images}"\n')
        f.write('mkdir -p "$OUTPUT_DIR"\n')
        f.write("\n")
        f.write("# Define source directory (UPDATE THIS)\n")
        f.write('SOURCE_DIR="/path/to/your/image/files"\n')
        f.write("\n")
        f.write("# Copy each file\n")
        f.write('echo "Starting file copy..."\n')
        f.write("COUNT=0\n")
        for filename in sorted(unique_filenames):
            f.write(f'if [ -f "$SOURCE_DIR/{filename}" ]; then\n')
            f.write(f'    cp "$SOURCE_DIR/{filename}" "$OUTPUT_DIR/"\n')
            f.write("    COUNT=$((COUNT + 1))\n")
            f.write('    echo "Copied: {filename}"\n'.format(filename=filename))
            f.write("else\n")
            f.write(f'    echo "WARNING: File not found: {filename}"\n')
            f.write("fi\n")
        f.write("\n")
        f.write('echo "Copy complete. Copied $COUNT files to $OUTPUT_DIR/"\n')

    # Make the bash script executable
    os.chmod(bash_script, 0o755)
    print(f"   Created bash script for file copying: {bash_script}")
    print(
        "   NOTE: You need to update SOURCE_DIR in the script to point to your actual image files."
    )

else:
    print("\nNo image data found for the specified patients.")

# Close connection
conn.close()

print("\n" + "=" * 80)
print("IMAGE FILENAME EXTRACTION COMPLETE")
print("=" * 80)

# Print quick summary of output files
if os.path.exists(output_dir):
    print(f"\nOutput files created in {output_dir}:")
    csv_files = [f for f in os.listdir(output_dir) if f.endswith(".csv")]
    txt_files = [f for f in os.listdir(output_dir) if f.endswith(".txt")]

    if csv_files:
        print("  CSV files:")
        for f in sorted(csv_files)[:5]:  # Show first 5
            print(f"    - {f}")
        if len(csv_files) > 5:
            print(f"    ... and {len(csv_files) - 5} more")

    if txt_files:
        print("  Text files:")
        for f in sorted(txt_files)[:5]:
            print(f"    - {f}")
        if len(txt_files) > 5:
            print(f"    ... and {len(txt_files) - 5} more")

EXTRACTING RAW IMAGE FILENAMES FOR SPECIFIED PATIENTS

1. Getting image filenames for PD patients (5 patients)...
   Found 809 unique image files for PD patients

2. Getting image filenames for HC patients (6 patients)...
   Found 486 unique image files for HC patients

3. Total unique image files found: 1295
   Saved all image data to: /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/raw_image_filenames_all.csv

4. Summary statistics:
   By disease:
disease  unique_images  unique_patients  total_oligomers
     HC            486                3         12314243
     PD            809                5         23861690

   By cell type:
disease cell_type  image_filename  oligomer_count_in_image
     HC microglia             243                  7115270
     HC   neurons             243                  5198973
     PD microglia             405                 13046807
     PD   neurons             404                 10814883

   By region:
disease    region

PermissionError: [Errno 1] Operation not permitted: '/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/copy_image_files.sh'

In [6]:
import duckdb
import pandas as pd
import numpy as np

# Connect to DuckDB
conn = duckdb.connect(database=":memory:")
parquet_path = "/scratch/duckdb-database/spot_analysis.parquet"

# ============================================================================
# USER INPUT: Define specific patients and cell types you want to analyze
# ============================================================================

# List of specific cell types you want to analyze
specific_cell_types = ["microglia", "neurons"]

# Define PD and HC patients (you need to define these lists)
# PD_patients = [...]  # Your PD patient IDs
# HC_patients = [...]  # Your HC patient IDs

# ============================================================================
# ANALYSIS SCRIPT
# ============================================================================

for i in range(2):
    if i == 0:
        specific_patients = PD_patients
        endstr = "PD"
    else:
        specific_patients = HC_patients
        endstr = "HC"

    print(f"\n{'='*80}")
    print(f"OLIGOMER ANALYSIS FOR {endstr} PATIENTS - CINGULATE CORTEX ONLY")
    print("=" * 80)

    # First, verify which of your specified patients exist in the database
    print(f"\n1. Checking for specified {endstr} patients in database:")
    print("-" * 40)

    patient_check_query = f"""
    SELECT DISTINCT donorid, disease 
    FROM '{parquet_path}' 
    WHERE donorid IN ({','.join([f"'{pid}'" for pid in specific_patients])})
      AND region = 'cingulate'
    ORDER BY donorid
    """

    existing_patients = conn.execute(patient_check_query).fetchdf()

    if existing_patients.empty:
        print(
            f"WARNING: None of the specified {endstr} patients were found in cingulate cortex!"
        )
        continue
    else:
        print(
            f"Found {len(existing_patients)} of {len(specific_patients)} specified patients in cingulate cortex:"
        )
        print(existing_patients.to_string(index=False))

        # Update the list to only include existing patients
        specific_patients = existing_patients["donorid"].tolist()

    print(f"\n2. Checking for specified cell types in cingulate cortex:")
    print("-" * 40)

    cell_type_check_query = f"""
    SELECT DISTINCT cell_type, COUNT(*) as total_oligomers
    FROM '{parquet_path}' 
    WHERE cell_type IN ({','.join([f"'{ct}'" for ct in specific_cell_types])})
      AND region = 'cingulate'
    GROUP BY cell_type
    ORDER BY total_oligomers DESC
    """

    existing_cell_types = conn.execute(cell_type_check_query).fetchdf()

    if existing_cell_types.empty:
        print(
            f"WARNING: None of the specified cell types were found in cingulate cortex!"
        )
        continue
    else:
        print(
            f"Found {len(existing_cell_types)} of {len(specific_cell_types)} specified cell types in cingulate cortex:"
        )
        print(existing_cell_types.to_string(index=False))

        # Update the list to only include existing cell types
        specific_cell_types = existing_cell_types["cell_type"].tolist()

    # ============================================================================
    # MAIN ANALYSIS: Oligomer counts per patient + cell type in CINGULATE ONLY
    # ============================================================================

    if specific_patients and specific_cell_types:
        print(
            f"\n3. Analyzing oligomer counts for {endstr} patients in CINGULATE CORTEX:"
        )
        print(f"   - {len(specific_patients)} patients")
        print(f"   - {len(specific_cell_types)} cell types")
        print("-" * 40)

        # Build the main query - FILTERED TO CINGULATE ONLY
        main_query = f"""
        SELECT 
            donorid,
            disease,
            cell_type,
            region,
            COUNT(*) as oligomer_count,
            -- Basic statistics
            AVG(sum_intensity_in_photons) as avg_intensity,
            STDDEV(sum_intensity_in_photons) as intensity_stddev,
            MIN(sum_intensity_in_photons) as min_intensity,
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sum_intensity_in_photons) as q1_intensity,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sum_intensity_in_photons) as median_intensity,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sum_intensity_in_photons) as q3_intensity,
            MAX(sum_intensity_in_photons) as max_intensity,
            -- Spatial distribution
            AVG(x) as avg_x,
            AVG(y) as avg_y,
            AVG(z) as avg_z
        FROM '{parquet_path}'
        WHERE donorid IN ({','.join([f"'{pid}'" for pid in specific_patients])})
          AND cell_type IN ({','.join([f"'{ct}'" for ct in specific_cell_types])})
          AND region = 'cingulate'
        GROUP BY donorid, disease, cell_type, region
        ORDER BY donorid, cell_type
        """

        # Execute the query
        results = conn.execute(main_query).fetchdf()

        if results.empty:
            print(
                f"No data found for the specified {endstr} patients and cell types in cingulate cortex!"
            )
        else:
            # ============================================================================
            # DISPLAY RESULTS
            # ============================================================================

            print(f"\n4. DETAILED RESULTS: {endstr} patients in CINGULATE CORTEX")
            print("-" * 80)

            # Display all results
            pd.set_option("display.max_rows", None)
            pd.set_option("display.max_columns", None)
            pd.set_option("display.width", None)

            print(results.to_string(index=False))

            # ============================================================================
            # CREATE SUMMARY TABLES
            # ============================================================================

            print(f"\n\n5. SUMMARY TABLES for {endstr} patients in CINGULATE")
            print("=" * 80)

            # A. Pivot table: Patients × Cell Types (oligomer counts)
            print("\nA. Total oligomers per patient × cell type (cingulate only):")
            pivot_total = results.pivot_table(
                index=["donorid", "disease"],
                columns="cell_type",
                values="oligomer_count",
                aggfunc="sum",
                fill_value=0,
            ).reset_index()

            # Add total column
            numeric_cols = pivot_total.select_dtypes(include=[np.number]).columns
            pivot_total["TOTAL"] = pivot_total[numeric_cols].sum(axis=1)

            # Sort by total
            pivot_total = pivot_total.sort_values("TOTAL", ascending=False)
            print(pivot_total.to_string(index=False))

            # B. Pivot table: Patients × Cell Types (average intensity)
            print("\nB. Average intensity per patient × cell type (cingulate only):")
            pivot_intensity = results.pivot_table(
                index=["donorid", "disease"],
                columns="cell_type",
                values="avg_intensity",
                aggfunc="mean",
                fill_value=0,
            ).reset_index()

            print(pivot_intensity.to_string(index=False))

            # C. Summary by cell type across all patients
            print(
                "\nC. Summary by cell type (cingulate only, across all specified patients):"
            )
            cell_type_summary = (
                results.groupby("cell_type")
                .agg(
                    {
                        "oligomer_count": ["sum", "mean", "std", "min", "max"],
                        "avg_intensity": ["mean", "std"],
                        "donorid": pd.Series.nunique,
                    }
                )
                .round(2)
            )

            # Flatten the multi-index columns
            cell_type_summary.columns = [
                "_".join(col).strip() for col in cell_type_summary.columns.values
            ]
            cell_type_summary = cell_type_summary.rename(
                columns={
                    "oligomer_count_sum": "total_oligomers",
                    "oligomer_count_mean": "avg_per_patient",
                    "oligomer_count_std": "std_per_patient",
                    "oligomer_count_min": "min_per_patient",
                    "oligomer_count_max": "max_per_patient",
                    "avg_intensity_mean": "avg_intensity",
                    "avg_intensity_std": "std_intensity",
                    "donorid_nunique": "patient_count",
                }
            ).reset_index()

            print(cell_type_summary.to_string(index=False))

            # D. Summary by patient across all cell types
            print(
                f"\nD. Summary by {endstr} patient (cingulate only, across all specified cell types):"
            )
            patient_summary = (
                results.groupby(["donorid", "disease"])
                .agg(
                    {
                        "oligomer_count": ["sum", "mean", "std", "min", "max"],
                        "avg_intensity": ["mean", "std"],
                        "cell_type": pd.Series.nunique,
                    }
                )
                .round(2)
            )

            patient_summary.columns = [
                "_".join(col).strip() for col in patient_summary.columns.values
            ]
            patient_summary = patient_summary.rename(
                columns={
                    "oligomer_count_sum": "total_oligomers",
                    "oligomer_count_mean": "avg_per_celltype",
                    "oligomer_count_std": "std_per_celltype",
                    "oligomer_count_min": "min_per_celltype",
                    "oligomer_count_max": "max_per_celltype",
                    "avg_intensity_mean": "avg_intensity",
                    "avg_intensity_std": "std_intensity",
                    "cell_type_nunique": "cell_type_count",
                }
            ).reset_index()

            patient_summary = patient_summary.sort_values(
                "total_oligomers", ascending=False
            )
            print(patient_summary.to_string(index=False))

            # ============================================================================
            # SAVE RESULTS TO FILES
            # ============================================================================

            print(f"\n\n6. SAVING RESULTS for {endstr} patients (cingulate only)")
            print("=" * 80)

            # Save all results
            results.to_csv(
                f"detailed_oligomer_counts_{endstr}_cingulate.csv", index=False
            )
            pivot_total.to_csv(
                f"patient_celltype_pivot_{endstr}_cingulate.csv", index=False
            )
            pivot_intensity.to_csv(
                f"patient_celltype_intensity_{endstr}_cingulate.csv", index=False
            )
            cell_type_summary.to_csv(
                f"celltype_summary_{endstr}_cingulate.csv", index=False
            )
            patient_summary.to_csv(
                f"patient_summary_{endstr}_cingulate.csv", index=False
            )

            # Also create a combined Excel file with multiple sheets
            with pd.ExcelWriter(
                f"oligomer_analysis_{endstr}_cingulate.xlsx", engine="openpyxl"
            ) as writer:
                results.to_excel(writer, sheet_name="Detailed_Data", index=False)
                pivot_total.to_excel(
                    writer, sheet_name="Patient_CellType_Counts", index=False
                )
                pivot_intensity.to_excel(
                    writer, sheet_name="Patient_CellType_Intensity", index=False
                )
                cell_type_summary.to_excel(
                    writer, sheet_name="CellType_Summary", index=False
                )
                patient_summary.to_excel(
                    writer, sheet_name="Patient_Summary", index=False
                )

            print(f"\nAll results saved with suffix: {endstr}_cingulate")
            print(f"CSV files:")
            print(f"  - detailed_oligomer_counts_{endstr}_cingulate.csv")
            print(f"  - patient_celltype_pivot_{endstr}_cingulate.csv")
            print(f"  - patient_celltype_intensity_{endstr}_cingulate.csv")
            print(f"  - celltype_summary_{endstr}_cingulate.csv")
            print(f"  - patient_summary_{endstr}_cingulate.csv")
            print(f"Excel file:")
            print(f"  - oligomer_analysis_{endstr}_cingulate.xlsx (all sheets)")

    else:
        print(
            f"\nERROR: No valid {endstr} patients or cell types to analyze in cingulate cortex!"
        )

# Close connection
conn.close()

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")


OLIGOMER ANALYSIS FOR PD PATIENTS - CINGULATE CORTEX ONLY

1. Checking for specified PD patients in database:
----------------------------------------
Found 5 of 5 specified patients in cingulate cortex:
donorid disease
 PD0086      PD
 PD0612      PD
 PD0913      PD
 PD0917      PD
 PD0945      PD

2. Checking for specified cell types in cingulate cortex:
----------------------------------------
Found 2 of 2 specified cell types in cingulate cortex:
cell_type  total_oligomers
  neurons         40668839
microglia         16307709

3. Analyzing oligomer counts for PD patients in CINGULATE CORTEX:
   - 5 patients
   - 2 cell types
----------------------------------------

4. DETAILED RESULTS: PD patients in CINGULATE CORTEX
--------------------------------------------------------------------------------
donorid disease cell_type    region  oligomer_count  avg_intensity  intensity_stddev  min_intensity  q1_intensity  median_intensity  q3_intensity  max_intensity      avg_x      avg_y    

In [ ]:
import duckdb
import pandas as pd

# Connect to DuckDB (you can use an in-memory database or connect to the existing one)
conn = duckdb.connect(database=":memory:", read_only=False)

# Method 1: Query the parquet file directly
print("Method 1: Querying parquet file directly")
try:
    # First, let's see the structure of the parquet file
    query = """
    SELECT * 
    FROM parquet_schema('/scratch/duckdb-database/spot_analysis.parquet')
    """
    schema = conn.execute(query).fetchdf()
    print("Parquet file schema:")
    print(schema.to_string())
except Exception as e:
    print(f"Error reading schema: {e}")

print("\n" + "=" * 80 + "\n")

# Method 2: Get column names and data types
print("Method 2: Getting column names and data types")
try:
    # This query gets column information
    query = """
    DESCRIBE SELECT * FROM '/scratch/duckdb-database/spot_analysis.parquet'
    """
    columns = conn.execute(query).fetchdf()
    print("Columns in the parquet file:")
    print(columns.to_string())
except Exception as e:
    print(f"Error describing columns: {e}")

print("\n" + "=" * 80 + "\n")

# Method 3: Sample some data to see structure
print("Method 3: Sampling data from the parquet file")
try:
    # Get first few rows to see structure
    query = """
    SELECT * 
    FROM '/scratch/duckdb-database/spot_analysis.parquet' 
    LIMIT 5
    """
    sample_data = conn.execute(query).fetchdf()
    print("First 5 rows of data:")
    print(sample_data.to_string())

    # Show shape
    count_query = """
    SELECT COUNT(*) as total_rows 
    FROM '/scratch/duckdb-database/spot_analysis.parquet'
    """
    total_rows = conn.execute(count_query).fetchone()[0]
    print(f"\nTotal rows in file: {total_rows:,}")
except Exception as e:
    print(f"Error sampling data: {e}")

print("\n" + "=" * 80 + "\n")

# Method 4: Check for specific columns like 'donorid'
print("Method 4: Checking for specific columns")
try:
    # Get all column names
    query = """
    SELECT column_name 
    FROM parquet_schema('/scratch/duckdb-database/spot_analysis.parquet')
    """
    column_names = [row[0] for row in conn.execute(query).fetchall()]

    print(f"Found {len(column_names)} columns")

    # Look for columns that might be donorid (case-insensitive)
    donorid_variants = ["donorid", "donorId", "DONORID", "donor_id", "DonorID"]
    found = False
    for variant in donorid_variants:
        for col in column_names:
            if col.lower() == variant.lower():
                print(f"Found column '{col}' (matches '{variant}')")
                found = True

                # Get unique values from this column
                unique_query = f"""
                SELECT DISTINCT "{col}" as donorid 
                FROM '/scratch/sycamore-asap/spot_analysis.parquet' 
                ORDER BY "{col}"
                LIMIT 10
                """
                unique_values = conn.execute(unique_query).fetchall()
                print(f"First 10 unique values: {[v[0] for v in unique_values]}")

                # Count unique values
                count_query = f"""
                SELECT COUNT(DISTINCT "{col}") as unique_count 
                FROM '/scratch/sycamore-asap/spot_analysis.parquet'
                """
                unique_count = conn.execute(count_query).fetchone()[0]
                print(f"Total unique values: {unique_count:,}")
                break

    if not found:
        print("No donorid column found. Here are all column names:")
        for i, col in enumerate(column_names, 1):
            print(f"  {i:3}. {col}")
except Exception as e:
    print(f"Error checking columns: {e}")

print("\n" + "=" * 80 + "\n")

# Method 5: Using pandas to read parquet metadata (alternative approach)
print("Method 5: Using pandas to read metadata")
try:
    # Read just the metadata (doesn't load all data)
    import pyarrow.parquet as pq

    parquet_file = pq.ParquetFile("/scratch/duckdb-database/spot_analysis.parquet")
    print(f"Parquet file metadata:")
    print(f"  Number of row groups: {parquet_file.num_row_groups}")
    print(f"  Number of rows: {parquet_file.metadata.num_rows}")
    print(f"  Created by: {parquet_file.metadata.created_by}")
    print(f"  Schema:")
    schema = parquet_file.schema
    for i in range(len(schema)):
        field = schema.field(i)
        print(f"    - {field.name}: {field.type}")

except Exception as e:
    print(f"Error with pandas/pyarrow: {e}")

conn.close()

In [4]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")
parquet_path = "/scratch/duckdb-database/spot_analysis.parquet"

# Get unique donorids with their disease
query = f"""
SELECT DISTINCT donorid, disease
FROM '{parquet_path}'
ORDER BY donorid
"""

try:
    donorid_disease = conn.execute(query).fetchdf()

    print("Unique donorids with disease status:")
    print("-" * 60)
    print(donorid_disease.to_string(index=False))

    # Count by disease
    print("\nCount by disease:")
    print(donorid_disease["disease"].value_counts())

    # Save to CSV
    donorid_disease.to_csv("donorids_with_disease.csv", index=False)
    print(f"\nSaved to 'donorids_with_disease.csv'")

except Exception as e:
    print(f"Error: {e}")

conn.close()

Unique donorids with disease status:
------------------------------------------------------------
donorid disease
   C030      HC
   C043      HC
   C045 unknown
   C046      HC
   C073      HC
   C074      HC
   C075      HC
   C076      HC
 PD0022      PD
 PD0086      PD
 PD0109      PD
 PD0590      PD
 PD0596      PD
 PD0612      PD
 PD0687      PD
 PD0779      PD
 PD0822      PD
 PD0913      PD
 PD0917      PD
 PD0945      PD
 PD0969      PD
 PD0980      PD
 PDC022      HC
 PDC029      HC
 PDC034      HC
 PDC035      HC
 PDC085      HC
 PDC087      HC
   None unknown

Count by disease:
disease
PD         14
HC         13
unknown     2
Name: count, dtype: int64

Saved to 'donorids_with_disease.csv'


In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")
parquet_path = "/scratch/duckdb-database/spot_analysis.parquet"

# Get unique donorids with their disease
query = f"""
SELECT DISTINCT donorid, cell_type
FROM '{parquet_path}'
ORDER BY donorid
"""

try:
    donorid_disease = conn.execute(query).fetchdf()

    print("Unique donorids with disease status:")
    print("-" * 60)
    print(donorid_disease.to_string(index=False))

    # Count by disease
    print("\nCount by cell type:")
    print(donorid_disease["cell_type"].value_counts())

    # Save to CSV
    donorid_disease.to_csv("donorids_with_celltypes.csv", index=False)
    print(f"\nSaved to 'donorids_with_celltypes.csv'")

except Exception as e:
    print(f"Error: {e}")

conn.close()

In [8]:
import duckdb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================================
# USER INPUT: Define specific patients you want to analyze
# ============================================================================

# Define your PD and HC patient lists (replace with your actual patient IDs)
PD_patients = ["PD0086", "PD0945", "PD0612", "PD0913", "PD0917"]
HC_patients = ["C030", "C064", "C036", "C043", "C049", "C074"]

# ============================================================================
# SETUP
# ============================================================================

percentile = 0  # Set the oligomer intensity percentile for the analysis
output_dir = (
    "/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots"
)
volume = (np.pi * (4 / 3)) * np.power(5, 3)

# Create a list to store all data for CSV output
all_data_for_csv = []

# ============================================================================
# FUNCTION TO CREATE PLOTS FOR SPECIFIC PATIENTS
# ============================================================================


def create_region_boxplot_for_patients(
    conn, cell_type, patient_list, disease_label, y_metric="incelldens"
):
    # Build patient list for SQL query
    patient_ids_str = ",".join([f"'{pid}'" for pid in patient_list])

    # Query only data needed for specific patients and cell type
    query = f"""
    SELECT 
        brain_region, 
        {y_metric}, 
        disease, 
        donorid
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type = '{cell_type}' 
        AND percentile = {percentile}
        AND "area/um3" >= 100 
        AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'parietal', 'frontal')
        AND donorid IN ({patient_ids_str})
    """

    subset = conn.execute(query).fetchdf()

    if subset.empty:
        print(
            f"No data for {cell_type} at {percentile}th percentile for {disease_label} patients"
        )
        return None

    # Store data for CSV output
    for index, row in subset.iterrows():
        all_data_for_csv.append(
            {
                "donorid": row["donorid"],
                "disease": row["disease"],
                "cell_type": cell_type,
                "brain_region": row["brain_region"],
                "incelldens": row[y_metric] * volume,
                "percentile": percentile,
            }
        )

    # Rename brain regions
    region_mapping = {
        "cingulate": "Cingulate",
        "frontal": "Frontal",
        "parietal": "Parietal",
    }
    subset["brain_region"] = subset["brain_region"].map(region_mapping)

    # Scale by volume
    subset[y_metric] = subset[y_metric] * volume

    # Define region order
    region_order = ["Parietal", "Cingulate", "Frontal"]

    # Set seaborn style
    sns.set_theme(style="ticks")

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # Create custom palette
    palette = {"PD": "#E88247", "HC": "#C3B3A1"}

    # Replace 'HC' with 'Control' in the dataset if needed
    if "HC" in subset["disease"].values:
        subset["disease"] = subset["disease"].replace("HC", "Control")
        palette = {"PD": "#E88247", "Control": "#C3B3A1"}

    # Create boxplot
    ax = sns.boxplot(
        data=subset,
        x="brain_region",
        y=y_metric,
        hue="disease",
        palette=palette,
        showfliers=False,  # Don't show outliers
        ax=ax,
        order=region_order,
    )

    # Add scatter points for individual donors with jittering
    for i, region in enumerate(region_order):
        region_data = subset[subset["brain_region"] == region]

        for j, disease in enumerate(["Control", "PD"]):
            disease_data = region_data[region_data["disease"] == disease]

            if disease_data.empty:
                continue

            donor_medians = (
                disease_data.groupby("donorid")[y_metric].median().reset_index()
            )

            if donor_medians.empty:
                continue

            # Calculate position for this box
            num_hues = 2  # PD and Control
            offset = 0.4
            pos_offset = (j - (num_hues - 1) / 2) * (offset / num_hues)
            x_pos_base = i + pos_offset

            for k, median_val in enumerate(donor_medians[y_metric]):
                # Add random jitter
                jitter = np.random.uniform(-0.05, 0.05)

                # Scatter plot for donor medians
                plt.scatter(
                    x_pos_base + jitter,
                    median_val,
                    color=palette[disease],
                    edgecolor="black",
                    s=30,
                    alpha=0.9,
                    zorder=10,
                )

    ax.tick_params(labelsize=7)

    # Create custom legend
    legend_elements = []
    for region in region_order:
        for disease in ["Control", "PD"]:
            region_disease_data = subset[
                (subset["brain_region"] == region) & (subset["disease"] == disease)
            ]
            if not region_disease_data.empty:
                unique_donors = region_disease_data["donorid"].nunique()
                cell_count = len(region_disease_data)
                region_short = (
                    "PARI"
                    if region == "Parietal"
                    else "CIN" if region == "Cingulate" else "FRO"
                )
                label = f"{disease} {region_short} (N = {unique_donors}; n_cells = {cell_count:,})"
                legend_elements.append(
                    Patch(facecolor=palette[disease], edgecolor="black", label=label)
                )

    # Add the custom legend
    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=6,
        ncols=1,
        bbox_to_anchor=(1.32, 1.05),
    )

    # Set labels and title
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.xticks(rotation=0, ha="right")

    # Set y-label
    if y_metric == "incelldens":
        plt.ylabel(f"αSyn oligomers per {cell_type}", fontsize=8)
    else:
        plt.ylabel(y_metric, fontsize=8)

    ax.set(xlabel=None)

    # Add title with patient info
    plt.title(
        f"{cell_type.capitalize()} - {disease_label} Patients (n={len(patient_list)})",
        fontsize=10,
    )

    # Save the plot
    filename = f"{output_dir}/{cell_type}_{disease_label}_{percentile}percentile_{y_metric}_by_region.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()

    print(f"Saved plot to {filename}")

    return subset


# ============================================================================
# MAIN ANALYSIS LOOP
# ============================================================================

# Reconnect to the database
conn = duckdb.connect("/scratch/duckdb-database/main_survey_database.duckdb")

# Define cell types to analyze
cell_types = ["microglia", "neurons"]

# Loop through PD and HC patients
for disease_label, patient_list in [("PD", PD_patients), ("HC", HC_patients)]:
    if not patient_list:
        print(f"No patients specified for {disease_label}")
        continue

    print(f"\n{'='*80}")
    print(f"ANALYZING {disease_label} PATIENTS")
    print(f"{'='*80}")

    for cell_type in cell_types:
        print(f"\nCreating plots for {cell_type} in {disease_label} patients...")
        subset = create_region_boxplot_for_patients(
            conn, cell_type, patient_list, disease_label, y_metric="incelldens"
        )

        if subset is not None:
            # Print summary statistics
            print(f"  Total cells: {len(subset):,}")
            print(f"  Unique donors: {subset['donorid'].nunique()}")
            print(
                f"  Average incelldens: {subset['incelldens'].mean():.2f} ± {subset['incelldens'].std():.2f}"
            )

            # Print per-patient summary
            print(f"\n  Per-patient summary:")
            patient_summary = (
                subset.groupby("donorid")
                .agg({"incelldens": ["mean", "std", "count"]})
                .round(2)
            )
            print(patient_summary)

# Close the connection
conn.close()

# ============================================================================
# SAVE DATA TO CSV
# ============================================================================

if all_data_for_csv:
    # Create DataFrame from collected data
    df_all_data = pd.DataFrame(all_data_for_csv)

    # Save to CSV
    csv_filename = f"{output_dir}/density_per_patient_{percentile}percentile.csv"
    df_all_data.to_csv(csv_filename, index=False)
    print(f"\n{'='*80}")
    print(f"Saved density data for {len(df_all_data)} rows to {csv_filename}")

    # Create summary pivot table
    print("\nSummary pivot table (mean incelldens per donor):")
    pivot_table = df_all_data.pivot_table(
        index=["donorid", "disease", "cell_type"],
        columns="brain_region",
        values="incelldens",
        aggfunc="mean",
    ).reset_index()

    print(pivot_table.to_string(index=False))

    # Save pivot table to CSV
    pivot_filename = f"{output_dir}/density_summary_pivot_{percentile}percentile.csv"
    pivot_table.to_csv(pivot_filename, index=False)
    print(f"\nSaved pivot table to {pivot_filename}")

    # Create Excel file with multiple sheets
    excel_filename = f"{output_dir}/density_analysis_{percentile}percentile.xlsx"
    with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
        df_all_data.to_excel(writer, sheet_name="All_Data", index=False)
        pivot_table.to_excel(writer, sheet_name="Summary_Pivot", index=False)

        # Add per-patient summary
        patient_summary = (
            df_all_data.groupby(["donorid", "disease", "cell_type", "brain_region"])
            .agg({"incelldens": ["mean", "std", "count", "min", "max"]})
            .round(3)
        )
        patient_summary.columns = [
            "_".join(col).strip() for col in patient_summary.columns.values
        ]
        patient_summary = patient_summary.reset_index()
        patient_summary.to_excel(writer, sheet_name="Detailed_Summary", index=False)

    print(f"Saved comprehensive Excel file to {excel_filename}")
else:
    print(
        "\nNo data collected. Please check your patient lists and database connection."
    )

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


ANALYZING PD PATIENTS

Creating plots for microglia in PD patients...
Saved plot to /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/microglia_PD_0percentile_incelldens_by_region.svg
  Total cells: 725
  Unique donors: 5
  Average incelldens: 97.67 ± 86.40

  Per-patient summary:
        incelldens             
              mean    std count
donorid                        
PD0086       90.29  89.01   110
PD0612       67.03  49.74   206
PD0913       47.42  59.25   150
PD0917      145.50  94.18   100
PD0945      159.82  87.58   159

Creating plots for neurons in PD patients...
Saved plot to /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/neurons_PD_0percentile_incelldens_by_region.svg
  Total cells: 3,320
  Unique donors: 5
  Average incelldens: 131.70 ± 148.97

  Per-patient summary:
        incelldens              
              mean     std count
donorid                         
PD0086      129.93  151.80   611
PD0612      1

In [9]:
def create_combined_plot_for_cell_type(
    conn, cell_type, pd_patients, hc_patients, y_metric="incelldens"
):
    """Create a combined plot with both PD and HC patients"""
    # Combine patient lists
    all_patients = pd_patients + hc_patients
    patient_ids_str = ",".join([f"'{pid}'" for pid in all_patients])

    # Query data
    query = f"""
    SELECT 
        brain_region, 
        {y_metric}, 
        disease, 
        donorid
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type = '{cell_type}' 
        AND percentile = {percentile}
        AND "area/um3" >= 100 
        AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'parietal', 'frontal')
        AND donorid IN ({patient_ids_str})
    """

    subset = conn.execute(query).fetchdf()

    if subset.empty:
        print(f"No data for {cell_type} at {percentile}th percentile")
        return None

    # Rename brain regions
    region_mapping = {
        "cingulate": "Cingulate",
        "frontal": "Frontal",
        "parietal": "Parietal",
    }
    subset["brain_region"] = subset["brain_region"].map(region_mapping)

    # Scale by volume
    subset[y_metric] = subset[y_metric] * volume

    # Define region order
    region_order = ["Parietal", "Cingulate", "Frontal"]

    # Set seaborn style
    sns.set_theme(style="ticks")

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # Create custom palette and rename HC to Control
    palette = {"PD": "#E88247", "Control": "#C3B3A1"}
    hue_order = ["Control", "PD"]

    # Replace 'HC' with 'Control' in the dataset
    subset["disease"] = subset["disease"].replace("HC", "Control")

    # Create boxplot
    ax = sns.boxplot(
        data=subset,
        x="brain_region",
        y=y_metric,
        hue="disease",
        hue_order=hue_order,
        palette=palette,
        showfliers=False,
        ax=ax,
        order=region_order,
    )

    # Add scatter points for individual donors with jittering
    for i, region in enumerate(region_order):
        region_data = subset[subset["brain_region"] == region]

        for j, disease in enumerate(hue_order):
            disease_data = region_data[region_data["disease"] == disease]

            if disease_data.empty:
                continue

            donor_medians = (
                disease_data.groupby("donorid")[y_metric].median().reset_index()
            )

            if donor_medians.empty:
                continue

            # Calculate position for this box
            num_hues = len(hue_order)
            offset = 0.4
            pos_offset = (j - (num_hues - 1) / 2) * (offset / num_hues)
            x_pos_base = i + pos_offset

            for k, median_val in enumerate(donor_medians[y_metric]):
                jitter = np.random.uniform(-0.05, 0.05)

                plt.scatter(
                    x_pos_base + jitter,
                    median_val,
                    color=palette[disease],
                    edgecolor="black",
                    s=30,
                    alpha=0.9,
                    zorder=10,
                )

    ax.tick_params(labelsize=7)

    # Create custom legend
    legend_elements = []
    for region in region_order:
        for disease in hue_order:
            region_disease_data = subset[
                (subset["brain_region"] == region) & (subset["disease"] == disease)
            ]
            if not region_disease_data.empty:
                unique_donors = region_disease_data["donorid"].nunique()
                cell_count = len(region_disease_data)
                region_short = (
                    "PARI"
                    if region == "Parietal"
                    else "CIN" if region == "Cingulate" else "FRO"
                )
                label = f"{disease} {region_short} (N = {unique_donors}; n_cells = {cell_count:,})"
                legend_elements.append(
                    Patch(facecolor=palette[disease], edgecolor="black", label=label)
                )

    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=6,
        ncols=1,
        bbox_to_anchor=(1.32, 1.05),
    )

    # Set labels
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.xticks(rotation=0, ha="right")

    if y_metric == "incelldens":
        plt.ylabel(f"αSyn oligomers per {cell_type}", fontsize=8)
    else:
        plt.ylabel(y_metric, fontsize=8)

    ax.set(xlabel=None)

    # Add title
    plt.title(
        f"{cell_type.capitalize()} - All Patients (PD: {len(pd_patients)}, HC: {len(hc_patients)})",
        fontsize=10,
    )

    # Save the plot
    filename = f"{output_dir}/{cell_type}_combined_{percentile}percentile_{y_metric}_by_region.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()

    print(f"Saved combined plot to {filename}")

    return subset

In [13]:
import duckdb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os

# ============================================================================
# USER INPUT: Define specific patients you want to analyze
# ============================================================================

# Define your PD and HC patient lists (replace with your actual patient IDs)
PD_patients = ["PD0086", "PD0945", "PD0612", "PD0913", "PD0917"]
HC_patients = ["C030", "C064", "C036", "C043", "C049", "C074"]

# ============================================================================
# SETUP
# ============================================================================

percentile = 0  # Set the oligomer intensity percentile for the analysis
output_dir = (
    "/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots"
)
volume = (np.pi * (4 / 3)) * np.power(5, 3)

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Reconnect to the database
conn = duckdb.connect("/scratch/duckdb-database/main_survey_database.duckdb")

# ============================================================================
# FUNCTION TO CREATE COMBINED PLOTS (PD + HC TOGETHER)
# ============================================================================


def create_combined_plot(conn, cell_type, y_metric="incelldens"):
    # Combine patient lists
    all_patients = PD_patients + HC_patients
    if not all_patients:
        print(f"No patients specified for {cell_type}")
        return None

    patient_ids_str = ",".join([f"'{pid}'" for pid in all_patients])

    # Query data for combined plot
    query = f"""
    SELECT 
        brain_region, 
        {y_metric}, 
        disease, 
        donorid
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type = '{cell_type}' 
        AND percentile = {percentile}
        AND "area/um3" >= 100 
        AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'parietal', 'frontal')
        AND donorid IN ({patient_ids_str})
    """

    subset = conn.execute(query).fetchdf()

    if subset.empty:
        print(f"No data for {cell_type} at {percentile}th percentile for combined plot")
        return None

    # Rename brain regions
    region_mapping = {
        "cingulate": "Cingulate",
        "frontal": "Frontal",
        "parietal": "Parietal",
    }
    subset["brain_region"] = subset["brain_region"].map(region_mapping)

    # Scale by volume
    subset[y_metric] = subset[y_metric] * volume

    # Define region order
    region_order = ["Parietal", "Cingulate", "Frontal"]

    # Set seaborn style
    sns.set_theme(style="ticks")

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # Create custom palette and rename HC to Control
    palette = {"PD": "#E88247", "Control": "#C3B3A1"}
    hue_order = ["Control", "PD"]

    # Replace 'HC' with 'Control' in the dataset
    subset["disease"] = subset["disease"].replace("HC", "Control")

    # Create boxplot
    ax = sns.boxplot(
        data=subset,
        x="brain_region",
        y=y_metric,
        hue="disease",
        hue_order=hue_order,
        palette=palette,
        showfliers=False,
        ax=ax,
        order=region_order,
    )

    # Add scatter points for individual donors with jittering
    for i, region in enumerate(region_order):
        region_data = subset[subset["brain_region"] == region]

        for j, disease in enumerate(hue_order):
            disease_data = region_data[region_data["disease"] == disease]

            if disease_data.empty:
                continue

            donor_medians = (
                disease_data.groupby("donorid")[y_metric].median().reset_index()
            )

            if donor_medians.empty:
                continue

            # Calculate position for this box
            num_hues = len(hue_order)
            offset = 0.4
            pos_offset = (j - (num_hues - 1) / 2) * (offset / num_hues)
            x_pos_base = i + pos_offset

            for k, median_val in enumerate(donor_medians[y_metric]):
                jitter = np.random.uniform(-0.05, 0.05)

                plt.scatter(
                    x_pos_base + jitter,
                    median_val,
                    color=palette[disease],
                    edgecolor="black",
                    s=30,
                    alpha=0.9,
                    zorder=10,
                )

    ax.tick_params(labelsize=7)

    # Create custom legend
    legend_elements = []
    for region in region_order:
        for disease in hue_order:
            region_disease_data = subset[
                (subset["brain_region"] == region) & (subset["disease"] == disease)
            ]
            if not region_disease_data.empty:
                unique_donors = region_disease_data["donorid"].nunique()
                cell_count = len(region_disease_data)
                region_short = (
                    "PARI"
                    if region == "Parietal"
                    else "CIN" if region == "Cingulate" else "FRO"
                )
                label = f"{disease} {region_short} (N = {unique_donors}; n_cells = {cell_count:,})"
                legend_elements.append(
                    Patch(facecolor=palette[disease], edgecolor="black", label=label)
                )

    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=6,
        ncols=1,
        bbox_to_anchor=(1.32, 1.05),
    )

    # Set labels
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.xticks(rotation=0, ha="right")

    if y_metric == "incelldens":
        plt.ylabel(f"αSyn oligomers per {cell_type}", fontsize=8)
    else:
        plt.ylabel(y_metric, fontsize=8)

    ax.set(xlabel=None)

    # Add title
    plt.title(
        f"{cell_type.capitalize()} - All Patients (PD: {len(PD_patients)}, HC: {len(HC_patients)})",
        fontsize=10,
    )

    # Save the plot
    filename = f"{output_dir}/{cell_type}_combined_{percentile}percentile_{y_metric}_by_region.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()

    print(f"Saved combined plot to {filename}")

    return subset


# ============================================================================
# CREATE COMBINED PLOTS FOR BOTH CELL TYPES
# ============================================================================

print("\n" + "=" * 80)
print("CREATING COMBINED PLOTS (PD + HC TOGETHER)")
print("=" * 80)

cell_types = ["microglia", "neurons"]
combined_data = {}

for cell_type in cell_types:
    print(f"\nCreating combined plot for {cell_type}...")
    subset = create_combined_plot(conn, cell_type, y_metric="incelldens")
    if subset is not None:
        combined_data[cell_type] = subset

# ============================================================================
# COMPREHENSIVE DATA COLLECTION WITH OLIGOMER COUNTS
# ============================================================================

print("\n" + "=" * 80)
print("COLLECTING COMPREHENSIVE DATA WITH OLIGOMER COUNTS")
print("=" * 80)

# Format patient IDs for the query
all_patients = PD_patients + HC_patients

if all_patients:
    patient_ids_str = ",".join([f"'{pid}'" for pid in all_patients])

    # First, get density data from PCL_dens_percell_3D
    print(f"\nQuerying density data for {len(all_patients)} patients...")
    density_query = f"""
    SELECT 
        donorid,
        disease,
        cell_type,
        brain_region,
        incelldens,
        puncta_cell_likelihood,
        "area/um3",
        percentile
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type IN ('microglia', 'neurons')
        AND percentile = {percentile}
        AND "area/um3" >= 100 
        AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'parietal', 'frontal')
        AND donorid IN ({patient_ids_str})
    ORDER BY donorid, cell_type, brain_region
    """

    density_data = conn.execute(density_query).fetchdf()

    if not density_data.empty:
        print(f"Retrieved {len(density_data)} rows of density data")

        # Calculate oligomers per cell (incelldens * volume)
        density_data["oligomers_per_cell"] = density_data["incelldens"] * volume

        # Create summary at the patient-celltype-region level from density data
        density_summary = (
            density_data.groupby(["donorid", "disease", "cell_type", "brain_region"])
            .agg(
                {
                    "incelldens": ["mean", "std", "count"],
                    "oligomers_per_cell": ["mean", "sum", "std"],
                }
            )
            .reset_index()
        )

        # Flatten column names
        density_summary.columns = [
            "_".join(col).strip() for col in density_summary.columns.values
        ]
        density_summary = density_summary.rename(
            columns={
                "donorid_": "donorid",
                "disease_": "disease",
                "cell_type_": "cell_type",
                "brain_region_": "brain_region",
                "incelldens_mean": "avg_incelldens",
                "incelldens_std": "std_incelldens",
                "incelldens_count": "cell_count",
                "oligomers_per_cell_mean": "avg_oligomers_per_cell",
                "oligomers_per_cell_sum": "total_oligomers_from_density",
                "oligomers_per_cell_std": "std_oligomers_per_cell",
            }
        )

        # Now get actual oligomer counts from spot_analysis.parquet
        print("\nQuerying actual oligomer counts from spot_analysis.parquet...")
        parquet_conn = duckdb.connect(database=":memory:")
        parquet_path = "/scratch/duckdb-database/spot_analysis.parquet"

        # Get oligomer counts per patient, cell type, and region
        oligomer_count_query = f"""
        SELECT 
            donorid,
            cell_type,
            region as brain_region,
            COUNT(*) as total_oligomers,
            AVG(sum_intensity_in_photons) as avg_intensity,
            MIN(sum_intensity_in_photons) as min_intensity,
            MAX(sum_intensity_in_photons) as max_intensity,
            STDDEV(sum_intensity_in_photons) as std_intensity
        FROM '{parquet_path}'
        WHERE donorid IN ({patient_ids_str})
          AND cell_type IN ('microglia', 'neurons')
          AND region IN ('cingulate', 'parietal', 'frontal')
        GROUP BY donorid, cell_type, region
        ORDER BY donorid, cell_type, region
        """

        oligomer_counts = parquet_conn.execute(oligomer_count_query).fetchdf()
        parquet_conn.close()

        if not oligomer_counts.empty:
            print(
                f"Retrieved oligomer counts for {len(oligomer_counts)} patient-celltype-region combinations"
            )

            # Merge the oligomer counts with the density data
            merged_summary = pd.merge(
                density_summary,
                oligomer_counts,
                on=["donorid", "cell_type", "brain_region"],
                how="left",
            )

            # Calculate additional metrics
            merged_summary["avg_incelldens_scaled"] = (
                merged_summary["avg_incelldens"] * volume
            )

            # Calculate oligomers per cell from actual counts
            merged_summary["oligomers_per_cell_from_counts"] = (
                merged_summary["total_oligomers"] / merged_summary["cell_count"]
            )

            # Calculate efficiency ratio (density vs actual counts)
            merged_summary["density_to_count_ratio"] = merged_summary[
                "total_oligomers_from_density"
            ] / merged_summary["total_oligomers"].replace(0, np.nan)

            # Reorder columns for clarity
            column_order = [
                "donorid",
                "disease",
                "cell_type",
                "brain_region",
                "cell_count",
                "total_oligomers",
                "avg_intensity",
                "min_intensity",
                "max_intensity",
                "std_intensity",
                "avg_incelldens",
                "avg_incelldens_scaled",
                "avg_oligomers_per_cell",
                "oligomers_per_cell_from_counts",
                "total_oligomers_from_density",
                "density_to_count_ratio",
                "std_incelldens",
                "std_oligomers_per_cell",
            ]

            # Filter to only include columns that exist
            existing_columns = [
                col for col in column_order if col in merged_summary.columns
            ]
            merged_summary = merged_summary[existing_columns]

            # Save comprehensive summary
            summary_filename = f"{output_dir}/comprehensive_oligomer_analysis_{percentile}percentile.csv"
            merged_summary.to_csv(summary_filename, index=False)
            print(f"\nSaved comprehensive summary to {summary_filename}")

            # ============================================================================
            # CREATE SIMPLIFIED SUMMARIES
            # ============================================================================

            print("\n" + "=" * 80)
            print("SUMMARY STATISTICS")
            print("=" * 80)

            # 1. Summary by disease and cell type
            print("\n1. Total oligomers by disease and cell type:")
            disease_cell_summary = (
                merged_summary.groupby(["disease", "cell_type"])
                .agg(
                    {
                        "total_oligomers": ["sum", "mean", "std", "min", "max"],
                        "cell_count": "sum",
                        "donorid": pd.Series.nunique,
                    }
                )
                .round(2)
            )

            # Flatten the multi-index columns
            disease_cell_summary.columns = [
                "_".join(col).strip() for col in disease_cell_summary.columns.values
            ]
            disease_cell_summary = disease_cell_summary.rename(
                columns={
                    "total_oligomers_sum": "total_oligomers",
                    "total_oligomers_mean": "avg_oligomers_per_combo",
                    "total_oligomers_std": "std_oligomers_per_combo",
                    "total_oligomers_min": "min_oligomers_per_combo",
                    "total_oligomers_max": "max_oligomers_per_combo",
                    "cell_count_sum": "total_cells",
                    "donorid_nunique": "patient_count",
                }
            ).reset_index()

            # Calculate oligomers per cell
            disease_cell_summary["oligomers_per_cell"] = (
                disease_cell_summary["total_oligomers"]
                / disease_cell_summary["total_cells"]
            )

            print(disease_cell_summary.to_string(index=False))

            # 2. Summary by brain region
            print("\n2. Total oligomers by brain region:")
            region_summary = (
                merged_summary.groupby(["brain_region", "cell_type"])
                .agg(
                    {
                        "total_oligomers": ["sum", "mean", "std"],
                        "cell_count": "sum",
                        "donorid": pd.Series.nunique,
                    }
                )
                .round(2)
            )

            region_summary.columns = [
                "_".join(col).strip() for col in region_summary.columns.values
            ]
            region_summary = region_summary.rename(
                columns={
                    "total_oligomers_sum": "total_oligomers",
                    "total_oligomers_mean": "avg_oligomers_per_combo",
                    "total_oligomers_std": "std_oligomers_per_combo",
                    "cell_count_sum": "total_cells",
                    "donorid_nunique": "patient_count",
                }
            ).reset_index()

            region_summary["oligomers_per_cell"] = (
                region_summary["total_oligomers"] / region_summary["total_cells"]
            )
            print(region_summary.to_string(index=False))

            # 3. Per-patient summary
            print("\n3. Per-patient summary (top 10 by total oligomers):")
            patient_summary = (
                merged_summary.groupby(["donorid", "disease"])
                .agg(
                    {
                        "total_oligomers": ["sum", "mean", "std", "min", "max"],
                        "cell_count": "sum",
                        "avg_intensity": "mean",
                    }
                )
                .round(2)
            )

            patient_summary.columns = [
                "_".join(col).strip() for col in patient_summary.columns.values
            ]
            patient_summary = patient_summary.rename(
                columns={
                    "total_oligomers_sum": "total_oligomers",
                    "total_oligomers_mean": "avg_oligomers_per_region",
                    "total_oligomers_std": "std_oligomers_per_region",
                    "total_oligomers_min": "min_oligomers_per_region",
                    "total_oligomers_max": "max_oligomers_per_region",
                    "cell_count_sum": "total_cells",
                    "avg_intensity_mean": "avg_intensity",
                }
            ).reset_index()

            patient_summary["oligomers_per_cell"] = (
                patient_summary["total_oligomers"] / patient_summary["total_cells"]
            )
            patient_summary = patient_summary.sort_values(
                "total_oligomers", ascending=False
            )
            print(patient_summary.head(10).to_string(index=False))

            # Save per-patient summary
            patient_summary_filename = (
                f"{output_dir}/per_patient_summary_{percentile}percentile.csv"
            )
            patient_summary.to_csv(patient_summary_filename, index=False)
            print(f"\nSaved per-patient summary to {patient_summary_filename}")

            # 4. Create simple pivot tables for Excel (without MultiIndex columns)
            print("\n4. Creating simplified pivot tables for Excel...")

            # Pivot 1: Total oligomers per patient × cell type (aggregated across regions)
            pivot_simple_total = (
                merged_summary.groupby(["donorid", "disease", "cell_type"])
                .agg({"total_oligomers": "sum", "cell_count": "sum"})
                .reset_index()
            )

            # Pivot to wide format
            pivot_total_wide = pivot_simple_total.pivot_table(
                index=["donorid", "disease"],
                columns="cell_type",
                values="total_oligomers",
                aggfunc="sum",
                fill_value=0,
            ).reset_index()

            # Add total column
            pivot_total_wide["Total_Oligomers"] = pivot_total_wide[
                ["microglia", "neurons"]
            ].sum(axis=1)

            # Pivot 2: Cells per patient × cell type
            pivot_cells_wide = pivot_simple_total.pivot_table(
                index=["donorid", "disease"],
                columns="cell_type",
                values="cell_count",
                aggfunc="sum",
                fill_value=0,
            ).reset_index()

            pivot_cells_wide["Total_Cells"] = pivot_cells_wide[
                ["microglia", "neurons"]
            ].sum(axis=1)

            # Pivot 3: Oligomers per cell
            pivot_oligomers_per_cell = pivot_simple_total.copy()
            pivot_oligomers_per_cell["oligomers_per_cell"] = (
                pivot_oligomers_per_cell["total_oligomers"]
                / pivot_oligomers_per_cell["cell_count"]
            )

            pivot_oligomers_per_cell_wide = pivot_oligomers_per_cell.pivot_table(
                index=["donorid", "disease"],
                columns="cell_type",
                values="oligomers_per_cell",
                aggfunc="mean",
                fill_value=0,
            ).reset_index()

            # ============================================================================
            # CREATE EXCEL FILE WITH MULTIPLE SHEETS
            # ============================================================================

            excel_filename = (
                f"{output_dir}/full_oligomer_analysis_{percentile}percentile.xlsx"
            )
            with pd.ExcelWriter(excel_filename, engine="openpyxl") as writer:
                # Sheet 1: All detailed data
                density_data.to_excel(writer, sheet_name="All_Cells_Data", index=False)

                # Sheet 2: Comprehensive summary
                merged_summary.to_excel(
                    writer, sheet_name="Comprehensive_Summary", index=False
                )

                # Sheet 3: Per-patient summary
                patient_summary.to_excel(
                    writer, sheet_name="Per_Patient_Summary", index=False
                )

                # Sheet 4: Disease-cell type summary
                disease_cell_summary.to_excel(
                    writer, sheet_name="Disease_Cell_Summary", index=False
                )

                # Sheet 5: Region summary
                region_summary.to_excel(
                    writer, sheet_name="Region_Summary", index=False
                )

                # Sheet 6: Total oligomers pivot
                pivot_total_wide.to_excel(
                    writer, sheet_name="Total_Oligomers_Pivot", index=False
                )

                # Sheet 7: Cell counts pivot
                pivot_cells_wide.to_excel(
                    writer, sheet_name="Cell_Counts_Pivot", index=False
                )

                # Sheet 8: Oligomers per cell pivot
                pivot_oligomers_per_cell_wide.to_excel(
                    writer, sheet_name="Oligomers_per_Cell_Pivot", index=False
                )

                # Sheet 9: By region detailed
                region_detailed = merged_summary.pivot_table(
                    index=["donorid", "disease", "cell_type"],
                    columns="brain_region",
                    values="total_oligomers",
                    aggfunc="sum",
                    fill_value=0,
                ).reset_index()
                region_detailed.to_excel(
                    writer, sheet_name="By_Region_Detailed", index=False
                )

            print(f"\nSaved comprehensive Excel file to {excel_filename}")

            # Generate a quick report
            report_filename = f"{output_dir}/analysis_report_{percentile}percentile.txt"
            with open(report_filename, "w") as f:
                f.write("OLIGOMER ANALYSIS REPORT\n")
                f.write("=" * 60 + "\n")
                f.write(f"Analysis date: {pd.Timestamp.now()}\n")
                f.write(f"Percentile analyzed: {percentile}\n")
                f.write(f"Total patients analyzed: {len(all_patients)}\n")
                f.write(f"  - PD patients: {len(PD_patients)}\n")
                f.write(f"  - HC patients: {len(HC_patients)}\n")
                f.write(f"Total cells analyzed: {density_data.shape[0]:,}\n")
                f.write(
                    f"Total oligomers detected: {merged_summary['total_oligomers'].sum():,}\n"
                )
                f.write(
                    f"Average oligomers per cell: {merged_summary['total_oligomers'].sum() / density_data.shape[0]:.2f}\n"
                )
                f.write("\n" + "=" * 60 + "\n")
                f.write("\nDISEASE SUMMARY:\n")
                f.write(disease_cell_summary.to_string(index=False))
                f.write("\n\nREGION SUMMARY:\n")
                f.write(region_summary.to_string(index=False))
                f.write("\n\nTOP 10 PATIENTS BY OLIGOMER COUNT:\n")
                f.write(patient_summary.head(10).to_string(index=False))

            print(f"\nSaved analysis report to {report_filename}")

            # Print key findings
            print("\n" + "=" * 80)
            print("KEY FINDINGS")
            print("=" * 80)
            print(
                f"1. Total oligomers across all patients: {merged_summary['total_oligomers'].sum():,}"
            )
            print(
                f"2. Average oligomers per cell: {merged_summary['total_oligomers'].sum() / density_data.shape[0]:.2f}"
            )
            print(
                f"3. Cell type with most oligomers: {disease_cell_summary.loc[disease_cell_summary['total_oligomers'].idxmax(), 'cell_type']}"
            )
            print(
                f"4. Region with most oligomers: {region_summary.loc[region_summary['total_oligomers'].idxmax(), 'brain_region']}"
            )
            print(
                f"5. Patient with most oligomers: {patient_summary.iloc[0]['donorid']} ({patient_summary.iloc[0]['total_oligomers']:,} oligomers)"
            )

        else:
            print("No oligomer count data found in spot_analysis.parquet")

            # Still save density data even without counts
            density_summary_filename = (
                f"{output_dir}/density_only_summary_{percentile}percentile.csv"
            )
            density_summary.to_csv(density_summary_filename, index=False)
            print(f"\nSaved density-only summary to {density_summary_filename}")

    else:
        print("No density data retrieved from the database")
else:
    print("No patients specified for analysis")

# Close connections
conn.close()

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


CREATING COMBINED PLOTS (PD + HC TOGETHER)

Creating combined plot for microglia...
Saved combined plot to /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/microglia_combined_0percentile_incelldens_by_region.svg

Creating combined plot for neurons...
Saved combined plot to /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/neurons_combined_0percentile_incelldens_by_region.svg

COLLECTING COMPREHENSIVE DATA WITH OLIGOMER COUNTS

Querying density data for 11 patients...
Retrieved 6932 rows of density data

Querying actual oligomer counts from spot_analysis.parquet...
Retrieved oligomer counts for 48 patient-celltype-region combinations

Saved comprehensive summary to /scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/20260119_ASAPPlots/comprehensive_oligomer_analysis_0percentile.csv

SUMMARY STATISTICS

1. Total oligomers by disease and cell type:
disease cell_type  total_oligomers  avg_oligomers_per_combo  std_oligomers_pe